In [10]:
"""
White House Presidential Actions Scraper — Selenium Edition (Fixed)
====================================================================
SETUP:
    pip install selenium webdriver-manager beautifulsoup4 pandas

RUN:
    python whitehouse_scraper_selenium.py

OUTPUT:
    whitehouse_presidential_actions_full.csv
"""

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
log = logging.getLogger(__name__)

BASE_URL = "https://www.whitehouse.gov/presidential-actions/page/{}/"

# ── CONFIG ────────────────────────────────────────────────────────────────────
MAX_PAGES = 50
HEADLESS   = True   # Set False to watch browser (helps debug)
DEBUG_HTML = True   # Set True to print a snippet of page HTML on first page
# ─────────────────────────────────────────────────────────────────────────────


def make_driver(headless: bool = True) -> webdriver.Chrome:
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opts
    )
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


def wait_for_page(driver, timeout=15):
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
    # Extra wait for JS-rendered content
    time.sleep(random.uniform(2.5, 4.0))


def parse_listing(soup: BeautifulSoup) -> list[dict]:
    """
    Try multiple selector strategies to find action items.
    Whitehouse.gov uses WordPress block patterns that may vary.
    """
    actions = []

    # Strategy 1: data-wp-key attribute (observed in DevTools screenshot)
    items = soup.find_all(
        "li",
        attrs={"data-wp-key": lambda v: v and v.startswith("post-template-item-")}
    )

    # Strategy 2: fallback — any <li> with a post class
    if not items:
        items = soup.find_all("li", class_=lambda c: c and "wp-block-post" in c)

    # Strategy 3: fallback — article tags
    if not items:
        items = soup.find_all("article")

    log.info(f"  Found {len(items)} raw item elements on page")

    for item in items:
        post_id = item.get("data-wp-key", "").replace("post-template-item-", "")

        # Title: try h2 first, then h3, then any heading
        title_tag = (
            item.find("h2", class_=lambda c: c and "post-title" in c)
            or item.find("h2")
            or item.find("h3")
            or item.find(["h1", "h2", "h3", "h4"])
        )
        title = title_tag.get_text(strip=True) if title_tag else ""

        # Link: try inside title tag, then any <a> in the item
        link_tag = (title_tag.find("a") if title_tag else None) or item.find("a")
        link = link_tag["href"] if link_tag and link_tag.has_attr("href") else ""

        # Categories
        terms_div = (
            item.find("div", class_=lambda c: c and "taxonomy-category" in c)
            or item.find("div", class_=lambda c: c and "post-terms" in c)
        )
        categories = []
        if terms_div:
            for a in terms_div.find_all("a"):
                cat = a.get_text(strip=True)
                if cat:
                    categories.append(cat)

        # Date
        date_display = date_iso = ""
        time_tag = item.find("time")
        if time_tag:
            date_display = time_tag.get_text(strip=True)
            date_iso = time_tag.get("datetime", "")

        if title or link:  # only add if we got something useful
            actions.append({
                "post_id":      post_id,
                "title":        title,
                "url":          link,
                "categories":   ", ".join(categories),
                "date_display": date_display,
                "date_iso":     date_iso,
                "content":      "",
            })

    return actions


def fetch_content(driver, url: str) -> str:
    """Navigate to action page and extract all <p> text."""
    if not url:
        return ""
    try:
        driver.get(url)
        wait_for_page(driver)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        paragraphs = soup.find_all("p")
        return "\n\n".join(
            p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)
        )
    except Exception as e:
        log.error(f"  ✗ Error fetching content from {url}: {e}")
        return ""


def scrape_all() -> pd.DataFrame:
    driver = make_driver(headless=HEADLESS)
    all_actions: list[dict] = []

    try:
        # ── Pass 1: listing pages ──
        for page_num in range(1, MAX_PAGES + 1):
            url = BASE_URL.format(page_num)
            log.info(f"[Listing] Page {page_num} → {url}")

            driver.get(url)
            wait_for_page(driver)

            # Log page title so we know what loaded
            log.info(f"  Page title: '{driver.title}'")
            log.info(f"  Current URL: {driver.current_url}")

            soup = BeautifulSoup(driver.page_source, "html.parser")

            # Debug: print HTML snippet on first page to inspect structure
            if page_num == 1 and DEBUG_HTML:
                body_text = soup.get_text()[:500]
                html_snippet = str(soup)[:2000]
                log.info(f"\n--- PAGE TEXT SNIPPET ---\n{body_text}\n---")
                log.info(f"\n--- HTML SNIPPET ---\n{html_snippet}\n---")

            # Stop only on true 404 (page_num > 1 and redirected back to page 1)
            if page_num > 1 and driver.current_url == BASE_URL.format(1):
                log.info("  Redirected to page 1 — end of pagination.")
                break

            actions = parse_listing(soup)
            if not actions and page_num > 1:
                log.info("  No items found — end of pagination.")
                break

            if actions:
                log.info(f"  → {len(actions)} actions collected")
                all_actions.extend(actions)
            else:
                log.warning("  No actions parsed on page 1 — check DEBUG output above!")
                log.warning("  The site structure may have changed. Try headless=False.")

            time.sleep(random.uniform(2.0, 4.0))

        if not all_actions:
            return pd.DataFrame()

        log.info(f"\nTotal actions to enrich: {len(all_actions)}")

        # ── Pass 2: full content ──
        for i, action in enumerate(all_actions, 1):
            log.info(f"[Content] {i}/{len(all_actions)} — {action['title'][:70]}...")
            action["content"] = fetch_content(driver, action["url"])
            time.sleep(random.uniform(1.5, 3.0))

            if i % 10 == 0:
                pd.DataFrame(all_actions[:i]).to_csv(
                    "whitehouse_actions_partial.csv", index=False, encoding="utf-8-sig"
                )
                log.info(f"  ✓ Checkpoint saved ({i} actions)")

    finally:
        driver.quit()
        log.info("Browser closed.")

    return pd.DataFrame(all_actions)


if __name__ == "__main__":
    df = scrape_all()

    if df.empty:
        print("\n❌ No data scraped.")
        print("\nTROUBLESHOOTING:")
        print("1. Set HEADLESS = False at the top of the script and re-run")
        print("   This lets you see what the browser is actually loading.")
        print("2. Check the '--- HTML SNIPPET ---' in the logs above.")
        print("   If it shows a Cloudflare challenge page, the site is heavily protected.")
        print("3. Try increasing the wait times (2.5 → 5.0) in wait_for_page().")
    else:
        out_file = "whitehouse_presidential_actions_full.csv"
        df.to_csv(out_file, index=False, encoding="utf-8-sig")
        print(f"\n✓ Done! Scraped {len(df)} actions → '{out_file}'")
        preview = df[["title", "categories", "date_display", "content"]].head(5).copy()
        preview["content"] = preview["content"].str[:100] + "..."
        print(preview.to_string(index=False))

2026-03-01 00:59:49,442 - ====== WebDriver manager ======
2026-03-01 00:59:49,611 - Get LATEST chromedriver version for google-chrome
2026-03-01 00:59:49,674 - Get LATEST chromedriver version for google-chrome
2026-03-01 00:59:49,734 - Driver [/Users/adithyakatari/.wdm/drivers/chromedriver/mac64/145.0.7632.117/chromedriver-mac-arm64/chromedriver] found in cache
2026-03-01 00:59:50,853 - [Listing] Page 1 → https://www.whitehouse.gov/presidential-actions/page/1/
2026-03-01 00:59:54,487 -   Page title: 'Presidential Actions – The White House'
2026-03-01 00:59:54,488 -   Current URL: https://www.whitehouse.gov/presidential-actions/
2026-03-01 00:59:54,523 - 
--- PAGE TEXT SNIPPET ---


















Presidential Actions – The White House























































































Skip to content






 



Menu














Search













Search for:
















Scroll Left




News
Gallery
Livestream
Investments
SAVE America
WH Wire



✓ Done! Scraped 483 actions → 'whitehouse_presidential_actions_full.csv'
                                                                                                                title                             categories      date_display                                                                                                     content
                                                                                      National Angel Family Day, 2026    Presidential Actions, Proclamations February 23, 2026 BY THE PRESIDENT OF THE UNITED STATES OF AMERICA\n\nA PROCLAMATION\n\nOn National Angel Family Day, we r...
                         Imposing a Temporary Import Surcharge to Address Fundamental International Payments Problems    Presidential Actions, Proclamations February 20, 2026 BY THE PRESIDENT OF THE UNITED STATES OF AMERICA\n\nA PROCLAMATION\n\nNOW, THEREFORE, I, DONALD J. TRUMP...
                                        Continuing the Suspension of Duty-Fre